# Import Data

* Perform selection in the mass range of 8.7 $\sigma$.
* Convert cosAngle to actual cos.
* Add left and right components to some variables to treat them separately.
* Add l over dl in xy plane.

In [1]:
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

import sys
import os

import comet_ml
import json


try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()
    
parent_dir = os.path.join(current_dir, '..', '..')
sys.path.insert(0, parent_dir)
    
    
from analysis_scripts.config import CFG


CFG.seed_all(CFG.seed)
sns.set_theme(style='ticks')

CUDA available: False
Use CPU, 
Number of CPUs: 12


In [2]:
data_oc_df = pd.read_csv('/home/ome123/Рабочий стол/SPD/SPD_Lc_pKpi_27/Data/analysed_signal_1.csv')
data_mb_df = pd.read_csv('/home/ome123/Рабочий стол/SPD/SPD_Lc_pKpi_27/Data/analysed_background_1.csv')

In [3]:
data_oc_df = data_oc_df.drop(['n_event'], axis=1)

mass_mask = (data_oc_df['mass_Lc'] > CFG.mass_interval[0]) \
            & ( data_oc_df['mass_Lc'] < CFG.mass_interval[1]) \
            & (data_oc_df['true_decay'] == 1)
            
data_oc_df = data_oc_df[mass_mask]
data_oc_df['tag'] = 'Sig'

data_mb_df = data_mb_df.drop(['n_event'], axis=1)

mass_mask = (data_mb_df['mass_Lc'] > CFG.mass_interval[0]) & ( data_mb_df['mass_Lc'] < CFG.mass_interval[1])

data_mb_df = data_mb_df[mass_mask]
data_mb_df['tag'] = 'Bg'

raw_df = pd.concat([data_oc_df, data_mb_df], axis=0).reset_index(drop=True)

raw_df['cosAngle_r_Lc_momentum_Lc_xy'] = raw_df['cosAngle_r_Lc_momentum_Lc_xy'].apply(np.cos)
raw_df['cosAngle_r_Lc_sum_momentum_xy'] = raw_df['cosAngle_r_Lc_sum_momentum_xy'].apply(np.cos)
raw_df['cosAngle_momentum_Lc_sum_momentum_xy'] = raw_df['cosAngle_momentum_Lc_sum_momentum_xy'].apply(np.cos)

raw_df['cosAngle_r_Lc_momentum_Lc_xy_left'] = raw_df.loc[raw_df['cosAngle_r_Lc_momentum_Lc_xy'] < 0, 'cosAngle_r_Lc_momentum_Lc_xy']
raw_df['cosAngle_r_Lc_momentum_Lc_xy_right'] = raw_df.loc[raw_df['cosAngle_r_Lc_momentum_Lc_xy'] > 0, 'cosAngle_r_Lc_momentum_Lc_xy']

raw_df['cosAngle_r_Lc_sum_momentum_xy_left'] = raw_df.loc[raw_df['cosAngle_r_Lc_sum_momentum_xy'] < 0, 'cosAngle_r_Lc_sum_momentum_xy']
raw_df['cosAngle_r_Lc_sum_momentum_xy_right'] = raw_df.loc[raw_df['cosAngle_r_Lc_sum_momentum_xy'] > 0, 'cosAngle_r_Lc_sum_momentum_xy']

raw_df['l_over_dl_XY'] = raw_df['lengthXY_Lc'] / raw_df['dlengthXY_Lc']

# Preselect Data

1. Preselection performs simple quantile-based selection. It cuts off all events that lie outside the quantile_left and quantile_right quantiles based on the signal distribution. The selection process begins if there are any signal events outside the interval:
(mean - safety_interval × indent; mean + safety_interval × indent).

2. Select only cases where the number of tracks selected for PV reconstruction is greater than min_tracks_after_refit.

In [4]:
from analysis_scripts.selection_scripts import auto_preselection


proc_df = raw_df.copy()

min_tracks_after_re_fit = 7

proc_df = proc_df[proc_df['kf_pv_size_after_re_fit'] > min_tracks_after_re_fit]

features_to_select = [
    'P_p', 'P_pip', 'P_K', 'P_Lc', 'eta_p', 'eta_pip',
       'eta_K', 'eta_Lc', 'Pt_Lc', 'Pt_p', 'Pt_K', 'Pt_pip', 'lengthXY_Lc',
       'dlengthXY_Lc', 'ctau_Lc', 'OA_p', 'OA_K', 'OA_pip', 'ptOverE',
       'chi2_Lc_PV_xy', 'dist_Lc_PV_xy', 'dist_Lc_PV_xy_custom',
       'chi2_p_PV_xy', 'dist_p_PV_xy', 'dist_p_PV_xy_custom', 'chi2_K_PV_xy',
       'dist_K_PV_xy', 'dist_K_PV_xy_custom', 'chi2_pip_PV_xy',
       'dist_pip_PV_xy', 'dist_pip_PV_xy_custom', 'chi2_p_Lc_xy',
       'dist_p_Lc_xy', 'dist_p_Lc_xy_custom', 'chi2_K_Lc_xy', 'dist_K_Lc_xy',
       'dist_K_Lc_xy_custom', 'chi2_pip_Lc_xy', 'dist_pip_Lc_xy',
       'dist_pip_Lc_xy_custom', 'chi2_Lc', 'chi2_K_pip_xy', 'dist_K_pip_xy',
       'dist_K_pip_xy_custom', 'chi2_p_K_xy', 'dist_p_K_xy',
       'dist_p_K_xy_custom', 'chi2_p_pip_xy', 'dist_p_pip_xy',
       'dist_p_pip_xy_custom', 'cosAngle_momentum_Lc_sum_momentum_xy',
       'xF', 'multiplicity', 'cosAngle_r_Lc_momentum_Lc_xy_left',
       'cosAngle_r_Lc_momentum_Lc_xy_right',
       'cosAngle_r_Lc_sum_momentum_xy_left',
       'cosAngle_r_Lc_sum_momentum_xy_right', 'l_over_dl_XY'
]

# Carefully handle _left, _right features
selection_df, mask = auto_preselection(
    df=proc_df,
    features=features_to_select,
    safety_interval=0.9,
    indent=2,
    quantile_left=1e-2,
    quantile_right=1-1e-2,
    recursive=True
)

proc_df = proc_df.query(mask)

proc_df_copy = proc_df.copy()

sig_eff_presel = proc_df.loc[proc_df['tag'] == 'Sig', 'mass_Lc'].count() / raw_df.loc[raw_df['tag'] == 'Sig', 'mass_Lc'].count()
bg_eff_presel = proc_df.loc[proc_df['tag'] == 'Bg', 'mass_Lc'].count() / raw_df.loc[raw_df['tag'] == 'Bg', 'mass_Lc'].count()

print(f'Signal efficiency: {sig_eff_presel}')
print(f'Background Suppression: {bg_eff_presel}')

Signal efficiency: 0.5005117308198415
Background Suppression: 0.15165792950603077


# Prepare data

In [16]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


features_list = CFG.features_set_9

df_train = proc_df[features_list + [CFG.target_name]].copy()
df_test = proc_df[features_list + [CFG.target_name]].copy()

x = df_train.drop('tag', axis=1).copy()
y = df_train['tag'].map({'Sig': 1, 'Bg': 0}).copy()

x_train, x_val, y_train, y_val = train_test_split(
    x, y, stratify=y, shuffle=True, test_size=0.2, random_state=CFG.seed
)

scaler = StandardScaler()
scaler.fit(x_train)
x_train = scaler.transform(x_train)
x_val = scaler.transform(x_val)

# Optuna

In [17]:
import optuna

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


def objective(trial):
    
    param_space = {
        'penalty': 'l2',
        'solver': 'newton-cholesky',
        'fit_intercept': True,
        'verbose': 0,
        'n_jobs': -1,
        'random_state': CFG.seed,
        
        'C': trial.suggest_float('C', 0.01, 100, log=True),
        'tol': trial.suggest_float('tol', 1e-6, 1e-1, log=True),
        'max_iter': trial.suggest_int('max_iter', 10, 1000),
    }

    # Var 1
    # local_x_train, local_x_val, local_y_train, local_y_val = train_test_split(
    #     x, y, stratify=y, shuffle=True, test_size=0.25
    # )
    # clf = XGBClassifier(**params).fit(local_x_train, local_y_train, eval_set=[(local_x_val, local_y_val)], verbose=0)
    # y_pred_proba = clf.predict_proba(local_x_val)[:, 1]
    # auc_score = roc_auc_score(local_y_val, y_pred_proba)
    # return {'score': auc_score, 'status': STATUS_OK}
    
    # Var 2
    seed = param_space['random_state'] # solve ModuleNotFoundError
    
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    split_generator = skf.split(x_train, y_train)
    
    train_scores = []
    val_scores = []
    for fold, (train_idx, val_idx) in enumerate(split_generator):
        
        fold_x_train = x_train[train_idx, :]
        fold_y_train = y_train.iloc[train_idx]
        fold_x_val = x_train[val_idx, :]
        fold_y_val = y_train.iloc[val_idx]
        
        fold_model = LogisticRegression(**param_space)
        
        fold_model.fit(fold_x_train, fold_y_train)
        
        fold_train_pred_proba = fold_model.predict_proba(fold_x_train)[:, 1]
        fold_val_pred_proba = fold_model.predict_proba(fold_x_val)[:, 1]
        
        fold_train_score = roc_auc_score(fold_y_train, fold_train_pred_proba)
        fold_val_score = roc_auc_score(fold_y_val, fold_val_pred_proba)
        
        train_scores.append(fold_train_score)
        val_scores.append(fold_val_score)

    train_scores = np.array(train_scores)
    val_scores = np.array(val_scores)

    trial.set_user_attr("train_scores_mean", train_scores.mean())

    return val_scores.mean()


study = optuna.create_study(
    direction='maximize', 
    study_name='log_reg_optimization',
    load_if_exists=True
)

study.optimize(objective, n_trials=100, timeout=320, n_jobs=-1, show_progress_bar=True)

[I 2026-03-01 15:43:01,838] A new study created in memory with name: log_reg_optimization


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-03-01 15:43:06,931] Trial 0 finished with value: 0.9309051191058393 and parameters: {'C': 0.6963715446890321, 'tol': 0.010433234062051627, 'max_iter': 449}. Best is trial 0 with value: 0.9309051191058393.
[I 2026-03-01 15:43:07,448] Trial 1 finished with value: 0.9309623571996124 and parameters: {'C': 4.068651842634092, 'tol': 0.00012644734934709104, 'max_iter': 945}. Best is trial 1 with value: 0.9309623571996124.
[I 2026-03-01 15:43:07,459] Trial 6 finished with value: 0.9309513703859356 and parameters: {'C': 0.08063168517244362, 'tol': 0.0037785285060886833, 'max_iter': 73}. Best is trial 1 with value: 0.9309623571996124.
[I 2026-03-01 15:43:07,511] Trial 11 finished with value: 0.9309494013218739 and parameters: {'C': 0.05804889871016234, 'tol': 3.447264830192308e-05, 'max_iter': 189}. Best is trial 1 with value: 0.9309623571996124.
[I 2026-03-01 15:43:07,565] Trial 9 finished with value: 0.9309597660242248 and parameters: {'C': 70.16488897708975, 'tol': 0.00410808492229703

In [18]:
best_trial = study.best_trial
print(f"Best validation score: {best_trial.value}")
print(f"Best train score: {best_trial.user_attrs['train_scores_mean']}")

best_params = study.best_params
print("Best parameters:", best_params)

Best validation score: 0.9309634474757548
Best train score: 0.9311352984399617
Best parameters: {'C': 0.304156318005697, 'tol': 1.1299866659534726e-06, 'max_iter': 821}


In [19]:
import plotly.graph_objects as go


trials_df = study.trials_dataframe()
val_scores = trials_df['value'].values
train_scores = [t.user_attrs['train_scores_mean'] for t in study.trials]

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(range(len(val_scores))),
    y=val_scores,
    mode='markers+lines',
    name='Validation Score',
    marker={'color': 'blue'}
))

fig.add_trace(go.Scatter(
    x=list(range(len(train_scores))),
    y=train_scores,
    mode='markers+lines',
    name='Train Score',
    marker={'color': 'red'}
))

fig.update_layout(
    title='Optimization History - Train vs Validation Scores',
    xaxis_title='Trial',
    yaxis_title='AUC Score',
    hovermode='x unified'
)

fig.show()

In [20]:
fig.write_html("sklearn_log_reg_features_set_9.html")